# 状态管理与持久化执行

## 基本环境准备

In [2]:
%pip install -U langchain langchain-openai langgraph-checkpoint-sqlite
%pip install -U python-dotenv
%pip install --upgrade pip
%pip install --no-build-isolation --config-settings="--global-option=build_ext" --config-settings="--global-option=-ID:\ruanjian\Graphviz\include" --config-settings="--global-option=-LD:\ruanjian\Graphviz\lib" pygraphviz


   ---------------------------------------- 3/3 [langgraph-checkpoint-sqlite]

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
from langchain.chat_models import init_chat_model
# === LLM 初始化（从 .env 加载） ===
import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("API_KEY")

model = init_chat_model(
    "LongCat-Flash-Chat",
    model_provider="openai",
    base_url="https://api.longcat.chat/openai",
    api_key=API_KEY,
    temperature=0.6
)

基本 State 定义

In [ ]:
from typing import TypedDict

class BasicState(TypedDict):
    """基本的 State 定义"""
    user_input: str
    response: str
    count: int

**使用 Annotated 和 Reducer**
- Annotated 是 Python 标准 typing 机制的一部分，用于给类型附加元数据。Python documentation
- 在 LangGraph 的场景下，它的元数据就是 reducer 函数（如 add、add_messages、lambda）。
- 类型检查器会看到基础类型（比如 int、list），而框架会看到元数据并以此驱动状态合并逻辑。

In [ ]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph.message import add_messages

class AdvancedState(TypedDict):
    # 普通字段：直接替换
    user_name: str
    session_id: str

    # 使用 add reducer：累加
    # 一个通用的 reducer 函数，对基础数据做“相加/累加”
    total_tokens: Annotated[int, add]

    # 使用 add_messages：消息列表管理
    # 一个专门处理消息（message lists）的 reducer
    # 对 消息列表（messages） 做追加 & 管理
    messages: Annotated[list, add_messages]

    # 自定义 reducer
    tags: Annotated[list, lambda old, new: list(set(old + new))]

## LangGraph Checkpoint

1. 是什么
- **Checkpoint = 存档**：保存 `state` 状态
- **MemorySaver**：内存存档，重启丢失，无需安装
- **thread_id**：存档编号，不同 ID 状态独立

2. 执行流程（最重要）
```
invoke 开始
  ↓
【读取检查点】（按 thread_id 读档，仅1次）
  ↓
运行节点1 → 保存检查点
  ↓
运行节点2 → 保存检查点
  ↓
结束
```

3. 关键规则
- **读取**：每次运行 **开头 1 次**
- **保存**：**每个节点跑完 1 次**
- 同 thread_id：状态延续
- 不同 thread_id：状态独立

4. 你的代码行为
- 第一次：`count=0 → 1`，存档 `1`
- 第二次：读档 `1` → 变成 `2`，存档 `2`
- 新 thread：重新从 `0` 开始

### 内存存储：MemorySaver
最简单的 Checkpoint 存储方式，将状态保存在内存中。

In [4]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class SimpleState(TypedDict):
    count: int

def increment(state: SimpleState) -> dict:
    return {"count": state.get("count", 0) + 1}

# 创建图
graph = StateGraph(SimpleState)
graph.add_node("increment", increment)
graph.add_edge(START, "increment")
graph.add_edge("increment", END)

# 使用 MemorySaver
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

# 运行时指定 thread_id
config = {"configurable": {"thread_id": "thread-1"}}

# 第一次运行
result1 = app.invoke({"count": 0}, config)
print(result1)  # {"count": 1}

# 第二次运行（同一个 thread，状态会累加）
result2 = app.invoke({}, config)
print(result2)  # {"count": 2}

# 新的 thread（独立的状态）
config2 = {"configurable": {"thread_id": "thread-2"}}
result3 = app.invoke({"count": 0}, config2)
print(result3)  # {"count": 1}

{'count': 1}
{'count': 2}
{'count': 1}


### SQLite 存储：SQLiteSaver

In [11]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# 创建 SQLite checkpointer
# checkpointer = SqliteSaver.from_conn_string("checkpoints.db")
checkpointer = SqliteSaver(sqlite3.connect("data/sqlite/checkpoints.db", check_same_thread=False))
# 或者使用内存 SQLite（用于测试）
# checkpointer = SqliteSaver(sqlite3.connect(":memory:", check_same_thread=False))

app = graph.compile(checkpointer=checkpointer)

# 使用方式与 MemorySaver 相同
config = {"configurable": {"thread_id": "user-123"}}
result = app.invoke({"count": 0}, config)
print(result)

{'count': 4}


## 实战示例：可恢复的数据管道处理

In [25]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
import time
import sqlite3
import os

# ==================== 全局变量（仅用于演示“仅第一次失败”） ====================
# 实际项目中请替换为业务逻辑（如数据库标记、Redis 锁、API 重试计数等）
batch2_failed_once = True

# ======================== State 定义 ========================
class PipelineState(TypedDict):
    total_items: int
    processed_items: Annotated[int, add]
    results: Annotated[list, lambda old, new: old + new]

# ======================== 幂等节点 ========================
def fetch_data(state: PipelineState) -> dict:
    """步骤 1：获取数据（幂等）"""
    current_total = state.get("total_items", 0)
    if current_total > 0:
        print("📥   数据已获取，跳过...")
        return {}
    print("📥   获取数据...")
    time.sleep(1)
    return {"total_items": 100}


def process_batch_1(state: PipelineState) -> dict:
    """步骤 2：处理批次 1（幂等）"""
    processed = state.get("processed_items", 0)
    if processed >= 33:
        print("⚙️   处理批次 1 (items 1-33) 已完成，跳过...")
        return {}
    print("⚙️   处理批次 1 (items 1-33)...")
    time.sleep(2)
    return {
        "processed_items": 33,
        "results": [f"result-{i}" for i in range(1, 34)]
    }


def process_batch_2(state: PipelineState) -> dict:
    """步骤 3：处理批次 2（幂等 + 仅第一次模拟失败）"""
    processed = state.get("processed_items", 0)
    if processed >= 66:
        print("⚙️   处理批次 2 (items 34-66) 已完成，跳过...")
        return {}

    print("⚙️   处理批次 2 (items 34-66)...")
    time.sleep(2)

    # ==================== 模拟瞬时错误（仅第一次执行失败） ====================
    global batch2_failed_once
    if processed == 33 and batch2_failed_once:
        batch2_failed_once = False
        raise Exception("💥   批次 2 处理失败！（模拟错误）")

    return {
        "processed_items": 33,
        "results": [f"result-{i}" for i in range(34, 67)]
    }


def process_batch_3(state: PipelineState) -> dict:
    """步骤 4：处理批次 3（幂等）"""
    processed = state.get("processed_items", 0)
    if processed >= 100:   # 66 + 34 = 100
        print("⚙️   处理批次 3 (items 67-100) 已完成，跳过...")
        return {}
    print("⚙️   处理批次 3 (items 67-100)...")
    time.sleep(2)
    return {
        "processed_items": 34,
        "results": [f"result-{i}" for i in range(67, 101)]
    }


def finalize(state: PipelineState) -> dict:
    """步骤 5：完成"""
    total_processed = state.get("processed_items", 0)
    print(f"✅   完成！总共处理 {total_processed} 项")
    return {}


# ======================== 构建图 ========================
graph = StateGraph(PipelineState)
graph.add_node("fetch", fetch_data)
graph.add_node("batch_1", process_batch_1)
graph.add_node("batch_2", process_batch_2)
graph.add_node("batch_3", process_batch_3)
graph.add_node("finalize", finalize)

graph.add_edge(START, "fetch")
graph.add_edge("fetch", "batch_1")
graph.add_edge("batch_1", "batch_2")
graph.add_edge("batch_2", "batch_3")
graph.add_edge("batch_3", "finalize")
graph.add_edge("finalize", END)

# ======================== SqliteSaver ========================
DB_PATH = "./data/sqlite/pipeline.db"
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
checkpointer = SqliteSaver(sqlite3.connect(DB_PATH, check_same_thread=False))
app = graph.compile(checkpointer=checkpointer)

# ======================== 执行演示 ========================
config = {"configurable": {"thread_id": "pipeline-6"}}

print("=== 第 1 次运行（模拟 batch_2 失败） ===")
initial_state = {
    "total_items": 0,
    "processed_items": 0,
    "results": []
}

try:
    result = app.invoke(initial_state, config)
except Exception as e:
    print(f"\n❌ 流程中断：{e}")

# ==================== 关键修复：从断点恢复 ====================
print("\n=== 第 2 次运行（从断点恢复） ===")
# 使用 None 表示“继续执行当前线程的下一个节点”
result = app.invoke({}, config)
result = app.invoke(None, config)
print(f"\n✅ 最终状态：{result}")

=== 第 1 次运行（模拟 batch_2 失败） ===
📥   获取数据...
⚙️   处理批次 1 (items 1-33) 已完成，跳过...
⚙️   处理批次 2 (items 34-66) 已完成，跳过...
⚙️   处理批次 3 (items 67-100) 已完成，跳过...
✅   完成！总共处理 100 项

=== 第 2 次运行（从断点恢复） ===
📥   数据已获取，跳过...
⚙️   处理批次 1 (items 1-33) 已完成，跳过...
⚙️   处理批次 2 (items 34-66) 已完成，跳过...
⚙️   处理批次 3 (items 67-100) 已完成，跳过...
✅   完成！总共处理 100 项

✅ 最终状态：{'total_items': 100, 'processed_items': 100, 'results': ['result-1', 'result-2', 'result-3', 'result-4', 'result-5', 'result-6', 'result-7', 'result-8', 'result-9', 'result-10', 'result-11', 'result-12', 'result-13', 'result-14', 'result-15', 'result-16', 'result-17', 'result-18', 'result-19', 'result-20', 'result-21', 'result-22', 'result-23', 'result-24', 'result-25', 'result-26', 'result-27', 'result-28', 'result-29', 'result-30', 'result-31', 'result-32', 'result-33', 'result-34', 'result-35', 'result-36', 'result-37', 'result-38', 'result-39', 'result-40', 'result-41', 'result-42', 'result-43', 'result-44', 'result-45', 'result-46', 'result-47'

### LangGraph `app.invoke(input, config)` 恢复机制对比文档

#### 1. 核心结论

| 调用方式                        | 含义                              | 是否可靠恢复（从失败点继续） | 是否需要节点**幂等** | 推荐场景 |
|-------------------------------|-----------------------------------|-----------------------------|---------------------|----------|
| `app.invoke(None, config)`    | **明确“继续执行”**（Resume）      | 是（推荐）                  | 不强制，但强烈建议   | 错误恢复、Human-in-the-loop、中断后继续 |
| `app.invoke({}, config)`      | **传入新输入**（可能视为新执行）  | 不一定，可能从头重跑         | **必须**幂等        | 正常新运行、需要额外输入时 |
| `app.invoke(initial_state, config)` | 传入初始状态，通常从头开始     | 基本从头开始                 | 必须幂等            | 第一次运行 |

---

#### 2. 为什么 `app.invoke({}, config)` 可能重头跑？

**根本原因**：

1. **LangGraph 对输入的处理逻辑**：
   - 当你传入**非 `None`** 的输入（包括空字典 `{}`）时，LangGraph 会认为你正在**提供新的用户输入**。
   - 它会从 `START` 节点开始执行，并尝试把这个输入应用到状态上。
   - 如果当前线程已经存在 checkpoint，它**不一定**会严格按照“从断点继续”的逻辑处理，而是可能触发**replay** 或**从头重建执行路径**。

2. **Checkpoint 保存时机**：
   - Checkpoint 是在**节点成功执行后**保存的（在 edge 跳转前）。
   - 如果 `batch_2` 抛异常，**本次节点的输出不会被 commit**，上一个 checkpoint 停留在 `batch_1` 之后。
   - 传入 `{}` 时，LangGraph 可能判断“这是一个新的 invocation”，从而从 `START` 或第一个 checkpoint 开始重新规划路径。

3. **历史版本行为差异**：
   - 早期 LangGraph 版本中，`invoke({})` 经常导致**完整 replay**（重头跑所有节点）。
   - 即使在新版本，**异常中断后**直接传 `{}` 仍然不够稳定，尤其在状态更新器（reducer）有累加逻辑时，容易造成重复处理。

**官方推荐做法**（来自 LangGraph 文档和大量示例）：
> “To resume the graph, pass `None` as input.”
> —— LangGraph Interrupts & Persistence 文档

---

#### 3. `app.invoke(None, config)` 的工作原理

- `None` 是**特殊信号**：表示“我没有新输入，请直接从当前线程的最后一个 checkpoint 继续执行下一个节点”。
- LangGraph 会：
  1. 加载该 `thread_id` 的最新 checkpoint。
  2. 从 `state.next`（下一个要执行的节点）继续运行。
  3. **不**重新执行已成功完成的节点（除非手动 time-travel）。

---

#### 4. 最佳实践总结（生产推荐）

```python
# 1. 第一次运行（可能失败）
try:
    result = app.invoke(initial_state, config)
except Exception as e:
    print("流程中断:", e)

# 2. 恢复运行（最稳方式）
result = app.invoke(None, config)   # ← 关键！
```

**额外强化建议**：

- **永远让节点保持幂等**（即使使用 `None` 也推荐），防止意外 replay 或并发问题。
- 使用 `interrupt_after=["batch_2"]` 代替异常来做暂停（更干净）。
- 调试时常用：
  ```python
  print(app.get_state(config))           # 当前在哪个节点
  print(app.get_state_history(config))   # 所有历史 checkpoint
  ```

---

#### 5. 一句话总结

- **`invoke(None, config)`** = “继续上次断掉的地方”（官方 resume 方式）
- **`invoke({}, config)`** = “给我一个新输入，可能从头开始”（普通执行方式）

因此，**错误恢复场景一定要用 `None`**，同时配合幂等节点才是最稳妥的生产方案。

---